In [2]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from multiprocessing import Pool, cpu_count
import os
from scipy.optimize import curve_fit
from sage.all import EllipticCurve, ZZ, primes_first_n, factor

OUTPUT_DIR = 'MW_torsion_curves/Paper_family_direct'
os.makedirs(OUTPUT_DIR, exist_ok=True)

#=======================
# Deshpande et al.'s paper family: y^2 = x^3 - D^2 x, D integer.
# Computing isogeny classes directly without Cremona database.
# D square-free gives all rational isomorphism classes.
#=======================

NUM_PRIMES = 1000
PRIMES = list(primes_first_n(NUM_PRIMES))

def is_squarefree(n):
    """Check if n is square-free."""
    for p, e in factor(abs(n)):
        if e >= 2:
            return False
    return True

def get_squarefree_Ds(D_max):
    """Get all square-free integers from 1 to D_max."""
    return [D for D in range(1, D_max + 1) if is_squarefree(D)]

def elliptic_curve_from_D(D):
    """Create elliptic curve y^2 = x^3 - D^2 x."""
    return EllipticCurve([ZZ(0), ZZ(0), ZZ(0), -ZZ(D)**2, ZZ(0)])

def process_D(D):
    """Process a single square-free D value. Returns dict or None."""
    try:
        E = elliptic_curve_from_D(D)
        E_min = E.minimal_model()
        N = int(E_min.conductor())
        
        # Compute isogeny class directly (no Cremona database needed)
        iso_class = E_min.isogeny_class()
        rep = iso_class.curves[0].minimal_model()
        iso_key = tuple(int(a) for a in rep.a_invariants())
        iso_size = len(iso_class.curves)
        
        # Rank computation
        rk = int(E_min.rank())
        
        # Frobenius traces
        ap_list = [int(E_min.ap(p)) for p in PRIMES]
        
        return {'D': D, 'conductor': N, 'iso_key': iso_key, 
                'iso_size': iso_size, 'rank': rk, 'ap_list': ap_list}
    except Exception as e:
        print(f"Error D={D}: {e}")
        return None

def power_law(x, A, alpha):
    return A / (x**alpha)

# ============================================================================
# SCAN AND PROCESS
# ============================================================================

D_max = 5000  # Adjust as needed; N ~ D^2 so D=5000 -> N ~ 25M
all_Ds = get_squarefree_Ds(D_max)
print(f"Processing {len(all_Ds)} square-free D values up to {D_max}...")

with Pool(processes=cpu_count()) as pool:
    results = list(tqdm(pool.imap(process_D, all_Ds), total=len(all_Ds)))

# Filter successful results
results = [r for r in results if r is not None]
print(f"Successfully processed {len(results)} curves")

# ============================================================================
# BUILD DATABASE
# ============================================================================

# Track all mappings
D_to_iso = {}           # D -> iso_key
iso_to_Ds = {}          # iso_key -> list of D values
iso_to_data = {}        # iso_key -> {conductor, ap_list, iso_size, D_rep}

for r in results:
    D, iso_key = r['D'], r['iso_key']
    D_to_iso[D] = iso_key
    
    if iso_key not in iso_to_Ds:
        iso_to_Ds[iso_key] = []
        iso_to_data[iso_key] = {
            'conductor': r['conductor'],
            'ap_list': r['ap_list'],
            'iso_size': r['iso_size'],
            'rank': r['rank'],
            'D_rep': D  # First D encountered as representative
        }
    iso_to_Ds[iso_key].append(D)

# Extract unique isogeny class data
isogeny_classes = list(iso_to_data.keys())
D_reps = [iso_to_data[iso]['D_rep'] for iso in isogeny_classes]
conductors = [iso_to_data[iso]['conductor'] for iso in isogeny_classes]
ap_lists = [iso_to_data[iso]['ap_list'] for iso in isogeny_classes]
iso_sizes = [iso_to_data[iso]['iso_size'] for iso in isogeny_classes]
ranks = [iso_to_data[iso]['rank'] for iso in isogeny_classes]

# Multiplicities: how many D values map to each isogeny class
multiplicities_by_iso = {iso: len(Ds) for iso, Ds in iso_to_Ds.items()}
multiplicities = [multiplicities_by_iso[D_to_iso[D]] for D in all_Ds if D in D_to_iso]

print(f"Found {len(isogeny_classes)} distinct isogeny classes")
print(f"Conductor range: [{min(conductors)}, {max(conductors)}]")

# ============================================================================
# COMPUTE AVERAGE FROBENIUS TRACES (OVERALL AND BY RANK)
# ============================================================================

ap_array = np.array(ap_lists)
avg_frob_traces = np.mean(ap_array, axis=0)
std_frob_traces = np.std(ap_array, axis=0)

# Separate by rank
rank_to_indices = {}
for i, rk in enumerate(ranks):
    if rk not in rank_to_indices:
        rank_to_indices[rk] = []
    rank_to_indices[rk].append(i)

avg_frob_by_rank = {}
std_frob_by_rank = {}
count_by_rank = {}

for rk, indices in rank_to_indices.items():
    rk_traces = np.array([ap_lists[i] for i in indices])
    avg_frob_by_rank[rk] = np.mean(rk_traces, axis=0)
    std_frob_by_rank[rk] = np.std(rk_traces, axis=0)
    count_by_rank[rk] = len(indices)
    print(f"Rank {rk}: {count_by_rank[rk]} isogeny classes")

print(f"Average Frobenius trace statistics computed over {len(ap_lists)} isogeny classes")

# ============================================================================
# SAVE DATA
# ============================================================================

print("Saving data...")
scan_data = {
    'all_Ds': all_Ds,
    'D_reps': D_reps,
    'isogeny_classes': isogeny_classes,
    'conductors': conductors,
    'ranks': ranks,
    'ap_lists': ap_lists,
    'iso_sizes': iso_sizes,
    'D_to_iso': D_to_iso,
    'iso_to_Ds': iso_to_Ds,
    'iso_to_data': iso_to_data,
    'multiplicities_by_iso': multiplicities_by_iso,
    'avg_frob_traces': avg_frob_traces.tolist(),
    'std_frob_traces': std_frob_traces.tolist(),
    'avg_frob_by_rank': {rk: arr.tolist() for rk, arr in avg_frob_by_rank.items()},
    'std_frob_by_rank': {rk: arr.tolist() for rk, arr in std_frob_by_rank.items()},
    'count_by_rank': count_by_rank,
    'primes': PRIMES,
    'D_max': D_max,
    'num_isogeny_classes': len(isogeny_classes),
    'num_Ds_processed': len(results)
}
save(scan_data, f'{OUTPUT_DIR}/D_max_{D_max}_scan_data.sobj')

# ============================================================================
# PLOTTING
# ============================================================================

print("Creating plots...")

# 1. Multiplicity vs D (for D values that were processed)
processed_Ds = [D for D in all_Ds if D in D_to_iso]
processed_mults = [multiplicities_by_iso[D_to_iso[D]] for D in processed_Ds]

plt.figure(figsize=(14, 7))
scatter = plt.scatter(processed_Ds, processed_mults, c=processed_mults, cmap='plasma',
                      alpha=0.6, s=15, edgecolors='none')
plt.colorbar(scatter, label='Multiplicity')
plt.xlabel(r'$D$', fontsize=14)
plt.ylabel('Multiplicity (# D → same isogeny class)', fontsize=14)
plt.title(fr'Isogeny class multiplicity vs $D$ (square-free $D \leq {D_max}$)', fontsize=16)
plt.grid(True, alpha=0.3, linestyle='--')
plt.axhline(y=1, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='Multiplicity = 1')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_multiplicity_vs_D.png', dpi=150)
plt.close()

# 2. D_rep distribution
plt.figure(figsize=(12, 6))
plt.hist(D_reps, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.xlabel(r'$D$ representative', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.title(fr'Distribution of $D$ representatives ({len(D_reps)} unique isogeny classes)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_D_rep_distribution.png', dpi=150)
plt.close()

# 3. Conductor vs D_rep
plt.figure(figsize=(12, 7))
plt.scatter(D_reps, conductors, c=np.log10(conductors), cmap='viridis', 
            alpha=0.6, s=20, edgecolors='none')
plt.colorbar(label=r'$\log_{10}(N)$')
plt.xlabel(r'$D$ representative', fontsize=14)
plt.ylabel(r'Conductor $N$', fontsize=14)
plt.title(fr'Conductor vs $D$ representative ($D \leq {D_max}$)', fontsize=16)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_conductor_vs_D.png', dpi=150)
plt.close()

# 4. Conductor distribution with power law fit
plt.figure(figsize=(14, 6))
counts_N, bins_N, _ = plt.hist(conductors, bins=100, density=True, alpha=0.6, label='Data')
bin_centers = (bins_N[:-1] + bins_N[1:]) / 2
mask = counts_N > 0

try:
    popt, _ = curve_fit(power_law, bin_centers[mask], counts_N[mask], p0=[1e6, 1.5], maxfev=10000)
    A_fit, alpha_fit = popt
    x_fit = np.linspace(min(conductors), max(conductors), 1000)
    plt.plot(x_fit, power_law(x_fit, A_fit, alpha_fit), 'r-', linewidth=2,
             label=fr'Power law: $\alpha={alpha_fit:.3f}$')
except Exception as e:
    print(f"Power law fit failed: {e}")

plt.xlabel(r"Conductor $N$", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.title(fr"Conductor distribution (square-free $D \leq {D_max}$)", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_conductor_distribution.png', dpi=150)
plt.close()

# 5. Isogeny class multiplicity histogram
mult_counts = {}
for m in multiplicities_by_iso.values():
    mult_counts[m] = mult_counts.get(m, 0) + 1

plt.figure(figsize=(12, 6))
mult_vals = sorted(mult_counts.keys())
mult_freqs = [mult_counts[m] for m in mult_vals]
plt.bar(mult_vals, mult_freqs, color='teal', edgecolor='black', alpha=0.7)
plt.xlabel('Multiplicity', fontsize=14)
plt.ylabel('Number of isogeny classes', fontsize=14)
plt.title(fr'Isogeny class multiplicities (square-free $D \leq {D_max}$)', fontsize=16)
plt.grid(True, alpha=0.3, linestyle='--', axis='y')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_multiplicity_histogram.png', dpi=150)
plt.close()

# 6. Average Frobenius traces by rank
plt.figure(figsize=(14, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(avg_frob_by_rank)))
for i, rk in enumerate(sorted(avg_frob_by_rank.keys())):
    plt.scatter(range(NUM_PRIMES), avg_frob_by_rank[rk], alpha=0.6, s=10, 
                color=colors[i], label=f'Rank {rk} (n={count_by_rank[rk]})')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.legend(fontsize=12)
plt.xlabel(r"Prime index $i$", fontsize=12)
plt.ylabel(r"Average Frobenius trace $\langle a_{p_i} \rangle$", fontsize=12)
plt.title(fr"Average Frobenius traces by rank ($D \leq {D_max}$)", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_avg_frobenius_traces_by_rank.png', dpi=150)
plt.close()

# 7. Overall average Frobenius traces
plt.figure(figsize=(14, 7))
plt.scatter(range(NUM_PRIMES), avg_frob_traces, alpha=0.6, s=10, 
            label=f'Average (n={len(ap_lists)} classes)')
plt.fill_between(range(NUM_PRIMES), 
                 avg_frob_traces - std_frob_traces, 
                 avg_frob_traces + std_frob_traces, 
                 alpha=0.2, label='±1 std dev')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.legend(fontsize=12)
plt.xlabel(r"Prime index $i$", fontsize=12)
plt.ylabel(r"Average Frobenius trace $\langle a_{p_i} \rangle$", fontsize=12)
plt.title(fr"Overall average Frobenius traces ($D \leq {D_max}$)", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_avg_frobenius_traces.png', dpi=150)
plt.close()

# 8. Rank 0 vs Rank 1 Frobenius traces
if 0 in avg_frob_by_rank and 1 in avg_frob_by_rank:
    plt.figure(figsize=(14, 7))
    plt.scatter(range(NUM_PRIMES), avg_frob_by_rank[0], alpha=0.6, s=12,
                color='blue', label=f'Rank 0 (n={count_by_rank[0]})')
    plt.scatter(range(NUM_PRIMES), avg_frob_by_rank[1], alpha=0.6, s=12,
                color='red', label=f'Rank 1 (n={count_by_rank[1]})')
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    plt.legend(fontsize=12)
    plt.xlabel(r"Prime index $i$", fontsize=12)
    plt.ylabel(r"Average Frobenius trace $\langle a_{p_i} \rangle$", fontsize=12)
    plt.title(fr"Rank 0 vs Rank 1: Average Frobenius traces ($D \leq {D_max}$)", fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_frobenius_rank0_vs_rank1.png', dpi=150)
    plt.close()
    
    # Normalized version
    plt.figure(figsize=(14, 7))
    plt.scatter(range(NUM_PRIMES), avg_frob_by_rank[0] / sqrt_primes, alpha=0.6, s=12,
                color='blue', label=f'Rank 0 (n={count_by_rank[0]})')
    plt.scatter(range(NUM_PRIMES), avg_frob_by_rank[1] / sqrt_primes, alpha=0.6, s=12,
                color='red', label=f'Rank 1 (n={count_by_rank[1]})')
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    plt.legend(fontsize=12)
    plt.xlabel(r"Prime index $i$", fontsize=12)
    plt.ylabel(r"$\langle a_{p_i} \rangle / \sqrt{p_i}$", fontsize=12)
    plt.title(fr"Rank 0 vs Rank 1: Normalized Frobenius traces ($D \leq {D_max}$)", fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_normalized_rank0_vs_rank1.png', dpi=150)
    plt.close()
else:
    print("Warning: Need both rank 0 and rank 1 curves for comparison plot")

# 10. Frobenius traces normalized by sqrt(p)
sqrt_primes = np.sqrt(PRIMES)
normalized_traces = avg_frob_traces / sqrt_primes

plt.figure(figsize=(14, 7))
plt.scatter(range(NUM_PRIMES), normalized_traces, alpha=0.6, s=10)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel(r"Prime index $i$", fontsize=12)
plt.ylabel(r"$\langle a_{p_i} \rangle / \sqrt{p_i}$", fontsize=12)
plt.title(fr"Normalized average Frobenius traces ($D \leq {D_max}$)", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_normalized_frobenius_traces.png', dpi=150)
plt.close()

# 11. Normalized Frobenius traces by rank
plt.figure(figsize=(14, 7))
for i, rk in enumerate(sorted(avg_frob_by_rank.keys())):
    normalized = avg_frob_by_rank[rk] / sqrt_primes
    plt.scatter(range(NUM_PRIMES), normalized, alpha=0.6, s=10,
                color=colors[i], label=f'Rank {rk} (n={count_by_rank[rk]})')
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.legend(fontsize=12)
plt.xlabel(r"Prime index $i$", fontsize=12)
plt.ylabel(r"$\langle a_{p_i} \rangle / \sqrt{p_i}$", fontsize=12)
plt.title(fr"Normalized average Frobenius traces by rank ($D \leq {D_max}$)", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_normalized_frobenius_by_rank.png', dpi=150)
plt.close()

# 12. Isogeny class size distribution
plt.figure(figsize=(12, 6))
size_counts = {}
for s in iso_sizes:
    size_counts[s] = size_counts.get(s, 0) + 1
sizes = sorted(size_counts.keys())
size_freqs = [size_counts[s] for s in sizes]
plt.bar(sizes, size_freqs, color='coral', edgecolor='black', alpha=0.7)
plt.xlabel('Isogeny class size', fontsize=14)
plt.ylabel('Number of isogeny classes', fontsize=14)
plt.title(fr'Isogeny class sizes ($D \leq {D_max}$)', fontsize=16)
plt.grid(True, alpha=0.3, linestyle='--', axis='y')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/D_max_{D_max}_isogeny_class_sizes.png', dpi=150)
plt.close()

print(f"\nSummary:")
print(f"  D_max: {D_max}")
print(f"  Square-free D values: {len(all_Ds)}")
print(f"  Successfully processed: {len(results)}")
print(f"  Unique isogeny classes: {len(isogeny_classes)}")
print(f"  Conductor range: [{min(conductors)}, {max(conductors)}]")
print(f"Done!")

Processing 3042 square-free D values up to 5000...


  3%|██▍                                                                              | 91/3042 [00:02<01:40, 29.51it/s]

Unable to compute the rank with certainty (lower bound=0).
This could be because Sha(E/Q)[2] is nontrivial.Unable to compute the rank with certainty (lower bound=0).

Try calling something like two_descent(second_limit=13) on theThis could be because Sha(E/Q)[2] is nontrivial.

curve then trying this command again.  You could also try rankTry calling something like two_descent(second_limit=13) on the

with only_use_mwrank=False.curve then trying this command again.  You could also try rank
Unable to compute the rank with certainty (lower bound=0).Error D=157: rank not provably correct (lower bound: 0)

This could be because Sha(E/Q)[2] is nontrivial.
with only_use_mwrank=False.

Error D=173: rank not provably correct (lower bound: 0)
Try calling something like two_descent(second_limit=13) on the
curve then trying this command again.  You could also try rank
with only_use_mwrank=False.
Error D=155: rank not provably correct (lower bound: 0)


  3%|██▌                                                                              | 95/3042 [00:03<02:28, 19.89it/s]

Unable to compute the rank with certainty (lower bound=0).
This could be because Sha(E/Q)[2] is nontrivial.
Try calling something like two_descent(second_limit=13) on the
curve then trying this command again.  You could also try rank
with only_use_mwrank=False.
Error D=203: rank not provably correct (lower bound: 0)


...

Unable to compute the rank with certainty (lower bound=0).
This could be because Sha(E/Q)[2] is nontrivial.
Try calling something like two_descent(second_limit=13) on the
curve then trying this command again.  You could also try rank
with only_use_mwrank=False.
Error D=4963: rank not provably correct (lower bound: 0)
Unable to compute the rank with certainty (lower bound=0).
This could be because Sha(E/Q)[2] is nontrivial.
Try calling something like two_descent(second_limit=13) on the
curve then trying this command again.  You could also try rank
with only_use_mwrank=False.
Error D=4955: rank not provably correct (lower bound: 0)


 99%|██████████████████████████████████████████████████████████████████████████████▎| 3016/3042 [01:54<00:01, 23.02it/s]

Unable to compute the rank with certainty (lower bound=0).
This could be because Sha(E/Q)[2] is nontrivial.
Try calling something like two_descent(second_limit=13) on the
curve then trying this command again.  You could also try rank
with only_use_mwrank=False.
Error D=4985: rank not provably correct (lower bound: 0)


100%|███████████████████████████████████████████████████████████████████████████████| 3042/3042 [01:55<00:00, 26.44it/s]


Successfully processed 2203 curves
Found 2203 distinct isogeny classes
Conductor range: [32, 797122592]
Rank 0: 1030 isogeny classes
Rank 1: 932 isogeny classes
Rank 2: 234 isogeny classes
Rank 3: 7 isogeny classes
Average Frobenius trace statistics computed over 2203 isogeny classes
Saving data...
Creating plots...

Summary:
  D_max: 5000
  Square-free D values: 3042
  Successfully processed: 2203
  Unique isogeny classes: 2203
  Conductor range: [32, 797122592]
Done!
